In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# SigLIP: TRAINING + EVALUATION

!pip install -q transformers datasets torch scikit-learn pillow

In [ ]:
# ---- IMPORTS ----
import torch, numpy as np
from transformers import SiglipProcessor, SiglipModel
from sklearn.metrics import accuracy_score, f1_score
from torch.nn.functional import normalize

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# ---- HF LOGIN ----
from huggingface_hub import login
login(token="your_token")


In [ ]:
# ---- LOAD DATASET ----
from datasets import load_dataset
ds = load_dataset("Navarasa-9/Navarasa_Balanced")

labels = sorted(set(ds["train"]["navarasa"]))
label2id = {l: i for i, l in enumerate(labels)}

ds = ds.map(lambda x: {"label": label2id[x["navarasa"]]})

README.md:   0%|          | 0.00/502 [00:00<?, ?B/s]

data/train-00000-of-00004.parquet:   0%|          | 0.00/420M [00:00<?, ?B/s]

data/train-00001-of-00004.parquet:   0%|          | 0.00/417M [00:00<?, ?B/s]

data/train-00002-of-00004.parquet:   0%|          | 0.00/425M [00:00<?, ?B/s]

data/train-00003-of-00004.parquet:   0%|          | 0.00/424M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/182M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6951 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/773 [00:00<?, ? examples/s]

Map:   0%|          | 0/6951 [00:00<?, ? examples/s]

Map:   0%|          | 0/773 [00:00<?, ? examples/s]

In [ ]:
# ---- METRICS ----
def recall_at_k(sim, k):
    return sum(i in sim[i].topk(k).indices for i in range(sim.size(0))) / sim.size(0)

def retrieval_metrics(img_emb, txt_emb):
    sim = img_emb @ txt_emb.T
    return {
        "R@1": recall_at_k(sim, 1),
        "R@5": recall_at_k(sim, 5),
        "R@10": recall_at_k(sim, 10),
    }

In [ ]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    images = [x["image"] for x in batch]
    texts  = [x["text"] for x in batch]
    labels = [x["label"] for x in batch]

    return images, texts, torch.tensor(labels)

In [ ]:
train_loader = DataLoader(
    ds["train"],
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn
)

In [ ]:
# ---- MODEL ----
processor = SiglipProcessor.from_pretrained("google/siglip-base-patch16-224")
model = SiglipModel.from_pretrained("google/siglip-base-patch16-224").to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/711 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/813M [00:00<?, ?B/s]

In [ ]:
!pip install -q tqdm

In [ ]:
from tqdm import tqdm

In [ ]:
num_epochs = 25

model.train()
for epoch in range(num_epochs):
    epoch_loss = 0.0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{num_epochs}",
        leave=True
    )

    for step, (images, texts, _) in enumerate(progress_bar):
        inputs = processor(
            images=images,
            text=texts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        outputs = model(**inputs, return_loss=True)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        epoch_loss += loss.item()

        # 🔹 update tqdm bar
        progress_bar.set_postfix({
            "step": step,
            "loss": f"{loss.item():.4f}"
        })

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Avg Loss: {epoch_loss / len(train_loader):.4f}"
    )

Epoch 1/25: 100%|██████████| 435/435 [04:40<00:00,  1.55it/s, step=434, loss=0.5490]


Epoch [1/25] Avg Loss: 1.9950


Epoch 2/25: 100%|██████████| 435/435 [04:37<00:00,  1.57it/s, step=434, loss=0.5470]


Epoch [2/25] Avg Loss: 1.1147


Epoch 3/25: 100%|██████████| 435/435 [04:37<00:00,  1.57it/s, step=434, loss=0.5675]


...

Epoch [23/25] Avg Loss: 0.4432


Epoch 24/25: 100%|██████████| 435/435 [04:38<00:00,  1.56it/s, step=434, loss=0.1271]


Epoch [24/25] Avg Loss: 0.4307


Epoch 25/25: 100%|██████████| 435/435 [04:37<00:00,  1.57it/s, step=434, loss=0.7627]

Epoch [25/25] Avg Loss: 0.4218


In [ ]:
siglip_save_dir = "/content/drive/MyDrive/models/siglip-balanced-navarasa"

model.save_pretrained(siglip_save_dir)
processor.save_pretrained(siglip_save_dir)

print("SigLIP model saved to Google Drive at:", siglip_save_dir)

SigLIP model saved to Google Drive at: /content/drive/MyDrive/models/siglip-balanced-navarasa


In [ ]:
!zip -r /content/siglip-balanced-navarasa.zip /content/drive/MyDrive/models/siglip-balanced-navarasa

  adding: content/drive/MyDrive/models/siglip-balanced-navarasa/ (stored 0%)
  adding: content/drive/MyDrive/models/siglip-balanced-navarasa/config.json (deflated 64%)
  adding: content/drive/MyDrive/models/siglip-balanced-navarasa/model.safetensors (deflated 7%)
  adding: content/drive/MyDrive/models/siglip-balanced-navarasa/preprocessor_config.json (deflated 50%)
  adding: content/drive/MyDrive/models/siglip-balanced-navarasa/tokenizer_config.json (deflated 63%)
  adding: content/drive/MyDrive/models/siglip-balanced-navarasa/special_tokens_map.json (deflated 72%)
  adding: content/drive/MyDrive/models/siglip-balanced-navarasa/spiece.model (deflated 49%)


In [ ]:
from google.colab import files

files.download("/content/siglip-balanced-navarasa.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ---- EVALUATE ----
model.eval()
img_emb, txt_emb, y_true, y_pred = [], [], [], []

with torch.no_grad():
    for s in ds["test"]:
        img = processor(images=s["image"], return_tensors="pt").to(device)
        txt = processor(text=s["text"], return_tensors="pt").to(device)

        i = normalize(model.get_image_features(**img), dim=-1)
        t = normalize(model.get_text_features(**txt), dim=-1)

        img_emb.append(i)
        txt_emb.append(t)

        y_pred.append((i @ t.T).argmax().item())
        y_true.append(s["label"])

img_emb = torch.cat(img_emb)
txt_emb = torch.cat(txt_emb)

print("\n[SigLIP Evaluation]")
print("Retrieval:", retrieval_metrics(img_emb, txt_emb))
print("Accuracy:", accuracy_score(y_true, y_pred))
print("F1 Macro:", f1_score(y_true, y_pred, average="macro"))
print("F1 Weighted:", f1_score(y_true, y_pred, average="weighted"))



[SigLIP Evaluation]
Retrieval: {'R@1': 0.2846054333764554, 'R@5': 0.7322121604139715, 'R@10': 0.8822768434670116}
Accuracy: 0.11384217335058215
F1 Macro: 0.025551684088269456
F1 Weighted: 0.023270873995008663
